# Synthetic valley: manual calibration of the base model

Let's calibrate the base model by hand: change one or more parameters, run the model, and see what happens to the fit. This notebook does that for the steady state of the base model of [`synthetic-valley-a-base`](synthetic-valley-a-base.ipynb), with four parameters on sliders:

- the horizontal K of the two layers **above the confining unit** (layers 1 and 2),
- the horizontal K of the two layers **below the confining unit** (layers 4 and 5),
- the **vertical K of the confining unit** (layer 3), and
- a **recharge multiplier** on every recharge cell.

Every slider move runs the model and compares its heads with the "true" steady-state heads at the 17 observation wells, in the water table and in the lower aquifer, on a one-to-one plot with the summary statistics, including phi - the sum of squared weighted residuals that PEST++ minimizes, and reports a prediction of interest: the steady-state discharge of groundwater to the stream.



## Imports

`sv_notebook_helpers` holds the location of the base model and `true_observations()`, which reads every observed series in the truth.

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sv_notebook_helpers import BASE_MODEL, true_observations

## Cut the model down to the steady state

The base model has 21 stress periods: a steady state followed by 20 years. Only the steady state is calibrated here, so the model keeps only its first stress period, which makes each run a fraction of the full model's. In the copy:

- the time discretization has one stress period, the steady state,
- the two well packages are removed, because their wells only pump from stress period 2,
- the recharge and evapotranspiration packages keep only their stress period 1 data, and
- the solver closes when heads change by less than 0.0001 m instead of the model's 0.000003 m, which saves iterations while changing heads by far less than the 0.1 m measurement noise.

The properties are also made **constant in each layer**: K, `k33`, specific storage, and specific yield each become one value per layer, the geometric mean of the layer's active cells, which MODFLOW 6 reads as a `CONSTANT` instead of an array. Every parameter below then sets one number in the NPF file. Making layer 1 constant removes the lake, which the base model represents by a very high K in its cells: those cells become aquifer like the rest of the layer, and only their higher recharge is left of the lake.

In [ ]:
name = "sv"
ws = pl.Path("models/sv-manual")  # the steady-state copy of the base model

sim = flopy.mf6.MFSimulation.load(
    sim_name=name, sim_ws=BASE_MODEL, write_headers=False, verbosity_level=0
)
sim.set_sim_path(ws)
gwf = sim.get_model(name)

# one stress period: the steady state
steady_state = sim.tdis.perioddata.array[0]
sim.tdis.nper = 1
sim.tdis.perioddata = [tuple(steady_state)]

# the wells only pump after the steady state
gwf.remove_package("pwell")
gwf.remove_package("prediction")

# recharge and evapotranspiration of stress period 1 only
nper = len(gwf.rch.stress_period_data.get_data())
for kper in range(1, nper):
    gwf.rch.stress_period_data.remove_transient_key(kper)
    gwf.evt.stress_period_data.remove_transient_key(kper)

# one value per layer of each property: the geometric mean of the layer's
# active cells, leaving out the lake cells of layer 1
aquifer = gwf.dis.idomain.array > 0
lake = gwf.npf.k.array[0] > 1000.0
aquifer[0] = aquifer[0] & ~lake


def layer_means(values):
    return [np.exp(np.log(values[k][aquifer[k]]).mean()) for k in range(len(values))]


k_layers = layer_means(gwf.npf.k.array)
k33_layers = layer_means(gwf.npf.k33.array)
gwf.npf.k.set_data(k_layers)
gwf.npf.k33.set_data(k33_layers)
gwf.sto.ss.set_data(layer_means(gwf.sto.ss.array))
gwf.sto.sy.set_data(layer_means(gwf.sto.sy.array))

# a looser closure than the model's, 0.0001 m, saves iterations and changes
# heads by far less than the noise
ims = sim.get_package("ims")
ims.complexity = "moderate"
ims.outer_dvclose = 1.0e-4
ims.outer_maximum = 500
ims.inner_dvclose = 1.0e-5

sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"
print((ws / "sv.npf").read_text())

## The observations

The observed values are the "true" steady-state heads at the 17 wells, in the water table (`wt`) and in the lower aquifer (`aq`), from the first output time of the truth. Each has a measurement standard deviation of 0.1 m, as in the PEST interface, and a weight of its inverse, 10, so that a residual of 0.1 m adds 1 to phi.

The **prediction** is the steady-state discharge of groundwater to the stream, the flow between the aquifer and the stream that MODFLOW 6 writes to `sv.riv.obs.csv`, negative when groundwater flows to the stream. It is not part of phi: the sliders are fitted to the heads, and the prediction shows what that fit implies for the stream. Its true value comes from the truth's flow between the aquifer and the stream.

In [ ]:
truth = true_observations()
t_steady = truth.index[0]  # the end of the steady-state stress period, days
wells = [c for c in truth.columns if c.startswith("wt") or c.startswith("aq")]
observed = truth.loc[t_steady, wells]
noise_sd = 0.1  # m
weight = 10.0  # of the steady-state heads in phi, the inverse of noise_sd
true_discharge = -truth.loc[t_steady, "riv-swgw"]  # to the stream, positive
print(f"{len(observed)} steady-state heads at day {t_steady:.0f}")
print(f"true discharge to the stream {true_discharge:.0f} m3/d")
observed.describe()

## The four parameters

`run_model()` sets the four parameters, writes the two packages they change, runs the model, and returns the simulated heads at the wells and the simulated discharge to the stream:

- **K above the confining unit** is the K of layers 1 and 2.
- **K below the confining unit** is the K of layers 4 and 5.
- **Vertical K of the confining unit.** The NPF package reads `k33` as the ratio of vertical to horizontal K (`K33OVERK`), so the ratio of layer 3 is set to the vertical K divided by its horizontal K. The vertical anisotropy of the other layers stays as it is.
- **The recharge multiplier** multiplies the recharge of every cell.

The starting values are the K of the layers (the K above the confining unit starts at the geometric mean of layers 1 and 2), the vertical K of the confining unit, and a multiplier of 1.

In [ ]:
base_rch = gwf.rch.stress_period_data.get_data(0).copy()
k3 = k_layers[2]  # the horizontal K of the confining unit stays as it is
start_k_upper = np.sqrt(k_layers[0] * k_layers[1])  # their geometric mean
start_k_lower = np.sqrt(k_layers[3] * k_layers[4])
start_kv_confining = k33_layers[2] * k3
print(f"K above the confining unit {start_k_upper:.1f} m/d")
print(f"K below the confining unit {start_k_lower:.1f} m/d")
print(f"vertical K of the confining unit {start_kv_confining:.4f} m/d")


def run_model(k_upper, k_lower, kv_confining, rch_multiplier):
    # one K for each layer, and the vertical anisotropy of layer 3
    gwf.npf.k.set_data([k_upper, k_upper, k3, k_lower, k_lower])
    gwf.npf.k33.set_data(
        [k33_layers[0], k33_layers[1], kv_confining / k3, k33_layers[3], k33_layers[4]]
    )

    rch = base_rch.copy()
    rch["recharge"] = base_rch["recharge"] * rch_multiplier
    gwf.rch.stress_period_data.set_data({0: rch})

    gwf.npf.write()
    gwf.rch.write()
    success, buff = sim.run_simulation(silent=True)
    assert success, "MODFLOW 6 did not terminate normally"

    wt = pd.read_csv(ws / "sv.gwf.wt.csv", index_col=0).iloc[0]
    aq = pd.read_csv(ws / "sv.gwf.aq.csv", index_col=0).iloc[0]
    simulated = pd.concat([wt, aq])
    simulated.index = simulated.index.str.lower()  # the truth's names
    riv = pd.read_csv(ws / "sv.riv.obs.csv", index_col=0).iloc[0]
    discharge = -riv["RIV-SWGW"]  # groundwater to the stream, positive
    return simulated.loc[observed.index], discharge

## Calibrate by hand

Each slider move runs the model, when the slider is released, and redraws the one-to-one plot: the simulated head at each well against the observed head, the water table in blue and the lower aquifer in orange. A perfect fit puts every point on the black line; the gray band is one measurement standard deviation either way. The box gives the summary statistics of the residuals (observed minus simulated): their number, mean, root mean square, and largest absolute value, and **phi**, the sum of squared weighted residuals. If every residual were as large as the noise, phi would be about the number of observations, 34. Below the statistics, the box gives the prediction, the simulated and true discharge of groundwater to the stream.

The K sliders are logarithmic, because K is usually uncertain by factors rather than by amounts.

In [ ]:
low = observed.min() - 1.0
high = observed.max() + 1.0
is_wt = observed.index.str.startswith("wt")


def show_fit(k_upper, k_lower, kv_confining, rch_multiplier):
    simulated, discharge = run_model(k_upper, k_lower, kv_confining, rch_multiplier)
    residual = observed - simulated
    phi = ((weight * residual) ** 2).sum()

    fig, ax = plt.subplots(figsize=(6.0, 6.0), layout="constrained")
    ax.fill_between(
        [low, high],
        [low - noise_sd, high - noise_sd],
        [low + noise_sd, high + noise_sd],
        color="0.85",
    )
    ax.plot([low, high], [low, high], color="black", lw=1.0)
    ax.plot(observed[is_wt], simulated[is_wt], "o", label="Water table (wt)")
    ax.plot(observed[~is_wt], simulated[~is_wt], "s", label="Lower aquifer (aq)")
    ax.set_xlim(low, high)
    ax.set_ylim(low, high)
    ax.set_aspect("equal")
    ax.set_xlabel("Observed head (m)")
    ax.set_ylabel("Simulated head (m)")
    ax.legend(loc="lower right")
    stats = (
        f"observations {len(residual)}\n"
        f"mean residual {residual.mean():.3f} m\n"
        f"RMS residual {np.sqrt((residual**2).mean()):.3f} m\n"
        f"largest |residual| {residual.abs().max():.3f} m\n"
        f"phi {phi:.1f}\n"
        f"\n"
        f"prediction: discharge to the stream\n"
        f"simulated {discharge:8.0f} m3/d\n"
        f"true      {true_discharge:8.0f} m3/d"
    )
    ax.text(0.03, 0.97, stats, transform=ax.transAxes, va="top", family="monospace")
    plt.show()


style = {"description_width": "200px"}
layout = widgets.Layout(width="600px")
controls = {
    "k_upper": widgets.FloatLogSlider(
        value=start_k_upper,
        min=0.0,
        max=3.0,
        step=0.02,
        description="K above confining unit (m/d)",
    ),
    "k_lower": widgets.FloatLogSlider(
        value=start_k_lower,
        min=0.0,
        max=3.0,
        step=0.02,
        description="K below confining unit (m/d)",
    ),
    "kv_confining": widgets.FloatLogSlider(
        value=start_kv_confining,
        min=-5.0,
        max=0.0,
        step=0.05,
        description="vertical K of confining unit (m/d)",
    ),
    "rch_multiplier": widgets.FloatLogSlider(
        value=1.0,
        min=-0.7,
        max=0.7,
        step=0.01,
        description="recharge multiplier",
    ),
}
for control in controls.values():
    control.style = style
    control.layout = layout
    control.continuous_update = False  # run the model when the slider is released
display(
    widgets.VBox(list(controls.values())),
    widgets.interactive_output(show_fit, controls),
)